# 05 — Free Agents

Explore the free agent pool: filtering by position, viewing stats, and identifying pickup targets.

In [ ]:
import sys
sys.path.insert(0, "../src")
from connection import get_league
import pandas as pd

league = get_league()
print(f"Connected: {league.settings.name}")

## Top Free Agents (all positions)

In [ ]:
fas = league.free_agents(size=50)
print(f"Retrieved {len(fas)} free agents\n")

rows = []
for p in fas:
    total_key = f"{league.year}_total"
    avg = p.stats.get(total_key, {}).get('avg', {})
    rows.append({
        "Name": p.name,
        "Pos": p.position,
        "NBA Team": p.proTeam,
        "Injury": p.injuryStatus,
        "Avg Pts": p.avg_points,
        "PTS": avg.get('PTS', 0),
        "REB": avg.get('REB', 0),
        "AST": avg.get('AST', 0),
        "STL": avg.get('STL', 0),
        "BLK": avg.get('BLK', 0),
        "3PM": avg.get('3PM', 0),
        "TO": avg.get('TO', 0),
    })

df = pd.DataFrame(rows)
df

## Filter by Position

Valid position strings: `'PG'`, `'SG'`, `'SF'`, `'PF'`, `'C'`, `'G'`, `'F'`

In [ ]:
for pos in ['PG', 'SG', 'SF', 'PF', 'C']:
    fas = league.free_agents(position=pos, size=5)
    names = [p.name for p in fas]
    print(f"{pos}: {', '.join(names)}")

## Free Agent Deep Dive

Inspect the full data on a single free agent.

In [ ]:
fas = league.free_agents(size=5)
fa = fas[0]

print(f"Free Agent: {fa.name}")
print(f"  playerId:      {fa.playerId}")
print(f"  position:      {fa.position}")
print(f"  eligibleSlots: {fa.eligibleSlots}")
print(f"  proTeam:       {fa.proTeam}")
print(f"  injuryStatus:  {fa.injuryStatus}")
print(f"  total_points:  {fa.total_points}")
print(f"  avg_points:    {fa.avg_points}")
print(f"  Stats keys:    {list(fa.stats.keys())}")

# Full stats dump
print(f"\nFull stats:")
for key, val in fa.stats.items():
    print(f"  {key}:")
    if isinstance(val, dict):
        for k2, v2 in val.items():
            display = str(v2)[:80]
            print(f"    {k2}: {display}")

## Streaming Category Targets

Find free agents who are strong in specific categories (useful for weekly streaming).

In [ ]:
fas = league.free_agents(size=100)

def get_avg_stat(player, stat):
    total_key = f"{league.year}_total"
    return player.stats.get(total_key, {}).get('avg', {}).get(stat, 0)

# Top free agents by each category
for cat in ['PTS', 'REB', 'AST', 'STL', 'BLK', '3PM']:
    top = sorted(fas, key=lambda p: get_avg_stat(p, cat), reverse=True)[:5]
    print(f"\nTop FA by {cat}:")
    for p in top:
        print(f"  {p.name:25s} {cat}: {get_avg_stat(p, cat):.1f}")